# Séance 8 · Exercices — Kaggle Titanic (2/2) : améliorer le score et éviter les pièges · ⭐⭐⭐

**Niveau : ⭐⭐⭐ Avancé**

**Niveau de la séance : ⭐⭐⭐ Avancé**

**Comment travailler** : lis l'énoncé, code dans la cellule `# À toi`, lance la cellule de vérification (✅ / ❌), et n'ouvre la solution qu'après avoir vraiment essayé.
Tout tourne dans **Google Colab** (rien à installer). Exécute chaque cellule avec `Maj + Entrée`, dans l'ordre : les exercices réutilisent ce qui précède.


## Quiz d'ouverture (2 min)

Réponds dans ta tête, puis déplie la réponse. Ce sont les questions qu'on se pose à voix haute au début de la séance.

**1. Un binôme annonce 98 % sur le Titanic. Réaction ?**
- a. Champagne, c'est le meilleur modèle du monde
- b. Méfiance : c'est trop beau, il y a sûrement une fuite ou une erreur
- c. Normal pour une forêt aléatoire
- d. Il faut soumettre tout de suite

<details><summary>Réponse</summary>

**b. Méfiance : c'est trop beau, il y a sûrement une fuite ou une erreur** — Sur le Titanic, un bon score tourne autour de 0,77 à 0,80. Au-delà de 0,85, on cherche l'erreur.

</details>


## Préparation

Mêmes données et même recette `preparer()` que la leçon, plus `X`, `y`, un tableau `X_suspect` pour l'exercice 9, et les fonctions `verifier()` (✅ / ❌) et `proche()` (compare deux nombres avec une tolérance).

In [ ]:
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, cross_validate, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline, Pipeline

sns.set_theme(style="whitegrid")

URL_SECOURS = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"

if os.path.exists("train.csv"):
    train = pd.read_csv("train.csv")
    print("train.csv de Kaggle chargé")
else:
    try:
        train = pd.read_csv(URL_SECOURS)
        print("Copie publique de train.csv chargée (mêmes colonnes que Kaggle)")
    except Exception as erreur:
        print("Impossible de charger les données : pas de réseau ?", erreur)
        raise
print(train.shape[0], "passagers,", train.shape[1], "colonnes")

# --- La recette de la leçon : regrouper_titre() et preparer() ---
def regrouper_titre(titre):
    """Garde les 4 titres fréquents, regroupe le reste dans « Autre »."""
    if titre in ["Mr", "Mrs", "Miss", "Master"]:
        return titre
    if titre in ["Mlle", "Ms"]:
        return "Miss"
    if titre == "Mme":
        return "Mrs"
    return "Autre"          # Dr, Rev, Col, Major, Countess, Capt...


COLONNES = ["Pclass", "Sex", "Age", "Fare", "Embarked", "Famille", "Seul", "Titre"]

def preparer(df):
    """Transforme le tableau brut de Kaggle en tableau de nombres prêt pour un modèle."""
    d = df.copy()
    # 1. Nouvelles variables
    d["Famille"] = d["SibSp"] + d["Parch"] + 1
    d["Seul"] = (d["Famille"] == 1).astype(int)
    d["Titre"] = d["Name"].str.extract(r",\s*([^\.]+)\.")[0].str.strip().apply(regrouper_titre)
    # 2. Cases vides : l'âge médian de chaque titre (un « Master » a 4 ans, un « Mr » 30)
    d["Age"] = d.groupby("Titre")["Age"].transform(lambda s: s.fillna(s.median()))
    d["Age"] = d["Age"].fillna(d["Age"].median())
    d["Fare"] = d["Fare"].fillna(d["Fare"].median())
    d["Embarked"] = d["Embarked"].fillna("S")
    # 3. Tout en nombres
    d["Sex"] = (d["Sex"] == "female").astype(int)
    d["Embarked"] = d["Embarked"].map({"S": 0, "C": 1, "Q": 2})
    d["Titre"] = d["Titre"].map({"Mr": 0, "Mrs": 1, "Miss": 2, "Master": 3, "Autre": 4})
    return d[COLONNES]

X = preparer(train)
y = train["Survived"]


# --- Pour l'exercice 9 : un tableau « suspect » avec 3 colonnes mystère (ne regarde pas comment il est fabriqué, joue le jeu !) ---
def _fabriquer_donnees_suspectes(X, y):
    rng = np.random.default_rng(0)
    d = X.copy()
    d["Code_A"] = rng.integers(0, 10, len(X))
    d["Code_B"] = np.where(rng.random(len(X)) < 0.9, y, 1 - y)
    d["Code_C"] = rng.normal(size=len(X)).round(2)
    return d


X_suspect = _fabriquer_donnees_suspectes(X, y)

# --- Vérification automatique : affiche ✅ ou ❌, ne plante jamais ---
def verifier(nom, condition):
    """condition : un booléen, ou une fonction sans argument qui renvoie un booléen."""
    try:
        ok = bool(condition() if callable(condition) else condition)
    except Exception:
        ok = False
    print(("✅ " if ok else "❌ ") + nom + ("" if ok else "  → pas encore, relis l'énoncé et réessaie"))


def proche(valeur, attendu, tolerance=0.01):
    """Vrai si valeur est un nombre à moins de `tolerance` de attendu."""
    try:
        return abs(float(valeur) - attendu) <= tolerance
    except (TypeError, ValueError):
        return False

print("Prêt !", X.shape, "- preparer(), X, y, X_suspect et verifier() sont chargés.")

## Exercice 1 ⭐ · Quatre modèles dans un tableau

Compare les 4 familles de la leçon avec la **même** validation croisée (5 paquets) et range les résultats dans un DataFrame `tableau` avec 3 colonnes : `Modèle`, `Précision moyenne`, `Écart-type`, trié de la meilleure précision à la moins bonne.

Le dictionnaire `modeles` est déjà rempli. Résultat attendu : 4 lignes, meilleur modèle ≈ `0.83` (gradient boosting ou forêt), le plus faible ≈ `0.80` (logistique). `meilleur_modele` = le nom en première ligne.

<details><summary>Indice</summary>

Boucle `for nom, modele in modeles.items():`, `scores = cross_val_score(modele, X, y, cv=5)`, ajoute un dict `{"Modèle": nom, ...}` à une liste, puis `pd.DataFrame(liste).sort_values("Précision moyenne", ascending=False)`.
</details>

In [ ]:
modeles = {
    "Régression logistique": LogisticRegression(max_iter=1000),
    "Arbre de décision": DecisionTreeClassifier(max_depth=4, random_state=42),
    "Forêt aléatoire": RandomForestClassifier(n_estimators=200, max_depth=5, random_state=42),
    "Gradient boosting": GradientBoostingClassifier(n_estimators=100, max_depth=3, random_state=42),
}

# À toi
tableau = None
meilleur_modele = None

tableau

In [ ]:
verifier("Exercice 1 · 4 lignes, 3 colonnes", lambda: tableau.shape == (4, 3) and list(tableau.columns) == ["Modèle", "Précision moyenne", "Écart-type"])
verifier("Exercice 1 · trié du meilleur au moins bon", lambda: tableau["Précision moyenne"].is_monotonic_decreasing)
verifier("Exercice 1 · meilleur ≈ 0.83, moins bon ≈ 0.80", lambda: proche(tableau["Précision moyenne"].max(), 0.832, 0.02) and proche(tableau["Précision moyenne"].min(), 0.8025, 0.02))
verifier("Exercice 1 · meilleur_modele", lambda: meilleur_modele == tableau.iloc[0]["Modèle"] and meilleur_modele in ("Gradient boosting", "Forêt aléatoire"))

<details><summary>Solution</summary>

```python
modeles = {
    "Régression logistique": LogisticRegression(max_iter=1000),
    "Arbre de décision": DecisionTreeClassifier(max_depth=4, random_state=42),
    "Forêt aléatoire": RandomForestClassifier(n_estimators=200, max_depth=5, random_state=42),
    "Gradient boosting": GradientBoostingClassifier(n_estimators=100, max_depth=3, random_state=42),
}

resultats = []
for nom, modele in modeles.items():
    scores = cross_val_score(modele, X, y, cv=5)
    resultats.append({"Modèle": nom, "Précision moyenne": scores.mean(), "Écart-type": scores.std()})

tableau = pd.DataFrame(resultats).sort_values("Précision moyenne", ascending=False).reset_index(drop=True)
meilleur_modele = tableau.iloc[0]["Modèle"]
print("Meilleur :", meilleur_modele)
tableau.round(3)
```
</details>

## Exercice 2 ⭐ · Stable ou capricieux ?

La moyenne ne dit pas tout : l'**écart-type** dit si le score change beaucoup d'un paquet à l'autre. À partir de `tableau` :
- `plus_stable` : le nom du modèle avec le plus petit écart-type
- `scores_plus_stable` : les 5 scores de ce modèle (relance `cross_val_score` sur `modeles[plus_stable]`)
- `ecart_meilleur_moins_bon` : différence entre la meilleure et la moins bonne précision moyenne du tableau

Résultat attendu : le modèle le plus stable a un écart-type < `0.01`, et les 4 modèles tiennent dans moins de `0.05` de précision : « un mouchoir de poche ».

<details><summary>Indice</summary>

`tableau.loc[tableau["Écart-type"].idxmin(), "Modèle"]` ou `tableau.sort_values("Écart-type").iloc[0]["Modèle"]`.
</details>

In [ ]:
# À toi
plus_stable = None
scores_plus_stable = None
ecart_meilleur_moins_bon = None

print("Le plus stable :", plus_stable, "- ses 5 scores :", scores_plus_stable)
print("Écart entre le meilleur et le moins bon modèle :", ecart_meilleur_moins_bon)

In [ ]:
verifier("Exercice 2 · plus_stable", lambda: plus_stable == tableau.loc[tableau["Écart-type"].idxmin(), "Modèle"] and plus_stable == "Régression logistique")
verifier("Exercice 2 · ses 5 scores varient peu", lambda: len(scores_plus_stable) == 5 and np.std(scores_plus_stable) < 0.01)
verifier("Exercice 2 · ecart_meilleur_moins_bon (< 0.05)", lambda: proche(ecart_meilleur_moins_bon, 0.029, 0.02) and ecart_meilleur_moins_bon > 0)

<details><summary>Solution</summary>

```python
plus_stable = tableau.loc[tableau["Écart-type"].idxmin(), "Modèle"]
scores_plus_stable = cross_val_score(modeles[plus_stable], X, y, cv=5)
ecart_meilleur_moins_bon = tableau["Précision moyenne"].max() - tableau["Précision moyenne"].min()

print("Le plus stable :", plus_stable, "- ses 5 scores :", scores_plus_stable.round(3))
print("Écart entre le meilleur et le moins bon modèle :", round(ecart_meilleur_moins_bon, 3))
# La logistique est la plus régulière (± 0,6 %) ; la forêt et le boosting gagnent en moyenne mais bougent plus.
```
</details>

## Exercice 3 ⭐ · Qui décide ? L'importance des variables

Entraîne `foret = RandomForestClassifier(n_estimators=200, max_depth=5, random_state=42)` sur `X, y`, puis :
- `importances` : Series `foret.feature_importances_` indexée par `COLONNES`, triée de la plus grande à la plus petite
- `top2` : liste des 2 variables les plus importantes
- `moins_utile` : le nom de la variable la moins utilisée

Trace un `barh`. Résultat attendu : les importances somment à `1`, `Titre` et `Sex` en tête (≈ 0.28 chacune), `Seul` en queue.

<details><summary>Indice</summary>

`pd.Series(foret.feature_importances_, index=COLONNES).sort_values(ascending=False)`, puis `list(importances.index[:2])` et `importances.index[-1]`.
</details>

In [ ]:
# À toi
foret = None
importances = None
top2 = None
moins_utile = None

print(importances)
print("Top 2 :", top2, "- la moins utile :", moins_utile)

In [ ]:
verifier("Exercice 3 · importances somment à 1", lambda: len(importances) == 8 and proche(importances.sum(), 1.0, 0.001))
verifier("Exercice 3 · triées décroissantes", lambda: importances.is_monotonic_decreasing)
verifier("Exercice 3 · top2 = Titre et Sex", lambda: set(top2) == {"Titre", "Sex"})
verifier("Exercice 3 · moins_utile = Seul", moins_utile == "Seul")

<details><summary>Solution</summary>

```python
foret = RandomForestClassifier(n_estimators=200, max_depth=5, random_state=42).fit(X, y)
importances = pd.Series(foret.feature_importances_, index=COLONNES).sort_values(ascending=False)
top2 = list(importances.index[:2])
moins_utile = importances.index[-1]

print(importances.round(3))
print("Top 2 :", top2, "- la moins utile :", moins_utile)

importances.sort_values().plot(kind="barh", figsize=(6, 3.5), color="tab:green")
plt.title("Quelles variables font la décision ?")
plt.show()
# Seul est redondant avec Famille (Seul = Famille == 1) : la forêt n'en a pas besoin.
```
</details>

## Exercice 4 ⭐ · La règle bête à battre

Avant de se réjouir d'un score, on le compare à la règle la plus simple :
- `prediction_bete` : Series de 0/1 qui vaut 1 pour les femmes et 0 pour les hommes (c'est le `gender_submission.csv` de Kaggle)
- `precision_bete` : proportion de passagers de `train` pour lesquels `prediction_bete == y`
- `precision_tout_le_monde_meurt` : la précision de la règle « personne ne survit » (prédire 0 partout)

Résultat attendu : `0.787` pour la règle « les femmes survivent », `0.616` pour « tout le monde meurt ». Ton modèle doit faire mieux que `0.787`, sinon il ne sert à rien.

<details><summary>Indice</summary>

`(train["Sex"] == "female").astype(int)` puis `(prediction_bete == y).mean()`. « Tout le monde meurt » a raison chaque fois que `y == 0`.
</details>

In [ ]:
# À toi
prediction_bete = None
precision_bete = None
precision_tout_le_monde_meurt = None

print("Femmes survivent :", precision_bete, "- tout le monde meurt :", precision_tout_le_monde_meurt)

In [ ]:
verifier("Exercice 4 · prediction_bete (314 survivantes prédites)", lambda: int(prediction_bete.sum()) == 314)
verifier("Exercice 4 · precision_bete ≈ 0.787", proche(precision_bete, 0.7868, 0.002))
verifier("Exercice 4 · precision_tout_le_monde_meurt ≈ 0.616", proche(precision_tout_le_monde_meurt, 0.6162, 0.002))

<details><summary>Solution</summary>

```python
prediction_bete = (train["Sex"] == "female").astype(int)
precision_bete = (prediction_bete == y).mean()
precision_tout_le_monde_meurt = (y == 0).mean()

print("Femmes survivent :", round(precision_bete, 3), "- tout le monde meurt :", round(precision_tout_le_monde_meurt, 3))
# Sur Kaggle la règle « femmes survivent » donne 0.76555 : c'est la barre à dépasser.
```
</details>

## Exercice 5 ⭐⭐ · La courbe du par cœur

Fais grandir un arbre de décision de la profondeur 1 à 12 et mesure deux scores à chaque fois avec `cross_validate(..., cv=5, return_train_score=True)` :
- `score_train` : liste des 12 précisions moyennes sur les données vues (`cv["train_score"].mean()`)
- `score_valid` : liste des 12 précisions moyennes sur les paquets cachés (`cv["test_score"].mean()`)
- `ecart_final` : `score_train[-1] - score_valid[-1]`, l'écart à la profondeur 12

Trace les deux courbes sur le même graphique. Résultat attendu : le train grimpe jusqu'à ≈ `0.96`, la validation plafonne autour de `0.82` puis redescend, `ecart_final` ≈ `0.18`.

<details><summary>Indice</summary>

`for p in profondeurs: cv = cross_validate(DecisionTreeClassifier(max_depth=p, random_state=42), X, y, cv=5, return_train_score=True)` puis `.append(...)` dans chaque liste. Graphique : `plt.plot(profondeurs, score_train, "o-", label="Train")`.
</details>

In [ ]:
profondeurs = list(range(1, 13))

# À toi
score_train = []
score_valid = []
ecart_final = None

print("Écart train - validation à la profondeur 12 :", ecart_final)

In [ ]:
verifier("Exercice 5 · 12 valeurs par courbe", lambda: len(score_train) == 12 and len(score_valid) == 12)
verifier("Exercice 5 · le train grimpe (> 0.94 à la fin)", lambda: score_train[-1] > 0.94 and score_train[-1] > score_train[0])
verifier("Exercice 5 · la validation plafonne ≈ 0.82", lambda: proche(max(score_valid), 0.8227, 0.02))
verifier("Exercice 5 · ecart_final > 0.12 : sur-apprentissage", lambda: proche(ecart_final, score_train[-1] - score_valid[-1], 1e-9) and ecart_final > 0.12)

<details><summary>Solution</summary>

```python
profondeurs = list(range(1, 13))

score_train = []
score_valid = []
for p in profondeurs:
    arbre = DecisionTreeClassifier(max_depth=p, random_state=42)
    cv = cross_validate(arbre, X, y, cv=5, return_train_score=True)
    score_train.append(cv["train_score"].mean())
    score_valid.append(cv["test_score"].mean())
ecart_final = score_train[-1] - score_valid[-1]

plt.figure(figsize=(8, 4))
plt.plot(profondeurs, score_train, "o-", label="Train (données vues)")
plt.plot(profondeurs, score_valid, "s-", label="Validation (données cachées)")
plt.xlabel("Profondeur de l'arbre")
plt.ylabel("Précision")
plt.legend()
plt.show()
print("Écart train - validation à la profondeur 12 :", round(ecart_final, 3))
```
</details>

## Exercice 6 ⭐⭐ · La bonne profondeur

À partir des listes de l'exercice 5 :
- `meilleure_profondeur` : la profondeur (un entier entre 1 et 12) où `score_valid` est maximal
- `arbre_final` : un `DecisionTreeClassifier` avec cette profondeur (`random_state=42`), entraîné sur `X, y`
- `nb_feuilles` : son nombre de feuilles (`arbre_final.get_n_leaves()`)

Résultat attendu : une profondeur de `4` (ou voisine), un arbre à quelques dizaines de feuilles au lieu de plusieurs centaines à la profondeur 12.

<details><summary>Indice</summary>

`int(np.argmax(score_valid))` donne la position du maximum (à partir de 0) : la profondeur correspondante est `profondeurs[position]`.
</details>

In [ ]:
# À toi
meilleure_profondeur = None
arbre_final = None
nb_feuilles = None

print("Meilleure profondeur :", meilleure_profondeur, "-", nb_feuilles, "feuilles")

In [ ]:
verifier("Exercice 6 · meilleure_profondeur = argmax de score_valid", lambda: meilleure_profondeur == profondeurs[int(np.argmax(score_valid))])
verifier("Exercice 6 · arbre_final entraîné avec cette profondeur", lambda: arbre_final.max_depth == meilleure_profondeur and arbre_final.get_depth() <= meilleure_profondeur)
verifier("Exercice 6 · nb_feuilles (moins de 40)", lambda: nb_feuilles == arbre_final.get_n_leaves() and nb_feuilles < 40)

<details><summary>Solution</summary>

```python
meilleure_profondeur = profondeurs[int(np.argmax(score_valid))]
arbre_final = DecisionTreeClassifier(max_depth=meilleure_profondeur, random_state=42).fit(X, y)
nb_feuilles = arbre_final.get_n_leaves()

print("Meilleure profondeur :", meilleure_profondeur, "-", nb_feuilles, "feuilles")
print("À la profondeur 12, le même arbre aurait", DecisionTreeClassifier(max_depth=12, random_state=42).fit(X, y).get_n_leaves(), "feuilles : une par petit groupe de passagers appris par cœur.")
```
</details>

## Exercice 7 ⭐⭐ · Un petit GridSearchCV

Plutôt que de tourner les boutons à la main, laisse `GridSearchCV` tester toutes les combinaisons d'une petite grille sur un arbre de décision :
- `grille = {"max_depth": [3, 5, 7], "min_samples_leaf": [1, 5, 10]}` (9 combinaisons)
- `recherche` : un `GridSearchCV(DecisionTreeClassifier(random_state=42), grille, cv=5)` entraîné sur `X, y`
- `meilleurs_reglages` : `recherche.best_params_`
- `meilleur_score` : `recherche.best_score_`

Résultat attendu : 9 combinaisons testées en moins d'une seconde, un score ≈ `0.82`.

<details><summary>Indice</summary>

`recherche.fit(X, y)` lance les 9 × 5 = 45 entraînements. `recherche.cv_results_["params"]` liste les combinaisons testées.
</details>

In [ ]:
grille = {"max_depth": [3, 5, 7], "min_samples_leaf": [1, 5, 10]}

# À toi
recherche = None
meilleurs_reglages = None
meilleur_score = None

print("Meilleurs réglages :", meilleurs_reglages, "- précision :", meilleur_score)

In [ ]:
verifier("Exercice 7 · recherche est un GridSearchCV entraîné", lambda: isinstance(recherche, GridSearchCV) and hasattr(recherche, "best_params_"))
verifier("Exercice 7 · 9 combinaisons testées", lambda: len(recherche.cv_results_["params"]) == 9)
verifier("Exercice 7 · meilleurs_reglages vient de la grille", lambda: meilleurs_reglages == recherche.best_params_ and meilleurs_reglages["max_depth"] in grille["max_depth"])
verifier("Exercice 7 · meilleur_score ≈ 0.82", proche(meilleur_score, 0.8215, 0.02))

<details><summary>Solution</summary>

```python
grille = {"max_depth": [3, 5, 7], "min_samples_leaf": [1, 5, 10]}

recherche = GridSearchCV(DecisionTreeClassifier(random_state=42), grille, cv=5)
debut = time.time()
recherche.fit(X, y)
meilleurs_reglages = recherche.best_params_
meilleur_score = recherche.best_score_

print(f"{len(recherche.cv_results_['params'])} combinaisons testées en {time.time() - debut:.1f} s")
print("Meilleurs réglages :", meilleurs_reglages, "- précision :", round(meilleur_score, 3))

# Toutes les combinaisons dans une grille de couleurs
res = pd.DataFrame(recherche.cv_results_)
carte = res.pivot(index="param_max_depth", columns="param_min_samples_leaf", values="mean_test_score")
sns.heatmap(carte.astype(float), annot=True, fmt=".3f", cmap="YlGn")
plt.title("Précision selon les 2 réglages")
plt.show()
```
</details>

## Exercice 8 ⭐⭐ · Le seuil de décision (predict_proba)

Un modèle ne répond pas « survit / meurt » : il donne une **probabilité**, et `predict` la coupe à `0.5`. Ce seuil se choisit.
1. Découpe `X, y` avec `train_test_split(test_size=0.25, random_state=42)` et entraîne une `LogisticRegression(max_iter=1000)`
2. `proba` : probabilité de survie de chaque passager de `X_test` (`predict_proba(X_test)[:, 1]`)
3. `nb_survivants_03`, `nb_survivants_05`, `nb_survivants_07` : nombre de passagers prédits survivants avec un seuil de 0.3, 0.5 et 0.7
4. `rappel_03` et `rappel_07` : parmi les **vrais** survivants de `y_test`, la proportion que chaque seuil attrape

Résultat attendu : `223` probabilités, environ `113`, `87` et `63` survivants prédits, et un rappel qui **baisse** quand le seuil monte (≈ `0.90` → `0.64`).

<details><summary>Indice</summary>

`(proba >= 0.3).astype(int)` donne les prédictions à ce seuil ; `.sum()` les compte. Le rappel : `predictions[y_test.values == 1].mean()`.
</details>

In [ ]:
# À toi
X_train, X_test, y_train, y_test = None, None, None, None
logistique = None
proba = None

nb_survivants_03 = None
nb_survivants_05 = None
nb_survivants_07 = None
rappel_03 = None
rappel_07 = None

print("Survivants prédits (seuil 0.3 / 0.5 / 0.7) :", nb_survivants_03, nb_survivants_05, nb_survivants_07)
print("Rappel (seuil 0.3 / 0.7) :", rappel_03, rappel_07)

In [ ]:
verifier("Exercice 8 · 223 probabilités entre 0 et 1", lambda: len(proba) == 223 and float(np.min(proba)) >= 0 and float(np.max(proba)) <= 1)
verifier("Exercice 8 · nb de survivants prédits ≈ 113 / 87 / 63", lambda: proche(nb_survivants_03, 113, 6) and proche(nb_survivants_05, 87, 6) and proche(nb_survivants_07, 63, 6))
verifier("Exercice 8 · plus le seuil monte, moins on prédit de survivants", lambda: nb_survivants_03 > nb_survivants_05 > nb_survivants_07)
verifier("Exercice 8 · le rappel baisse avec le seuil", lambda: proche(rappel_03, 0.90, 0.05) and proche(rappel_07, 0.64, 0.05) and rappel_03 > rappel_07)

<details><summary>Solution</summary>

```python
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
logistique = LogisticRegression(max_iter=1000).fit(X_train, y_train)
proba = logistique.predict_proba(X_test)[:, 1]

def predire_au_seuil(seuil):
    return (proba >= seuil).astype(int)

nb_survivants_03 = predire_au_seuil(0.3).sum()
nb_survivants_05 = predire_au_seuil(0.5).sum()
nb_survivants_07 = predire_au_seuil(0.7).sum()
rappel_03 = predire_au_seuil(0.3)[y_test.values == 1].mean()
rappel_07 = predire_au_seuil(0.7)[y_test.values == 1].mean()

print("Survivants prédits (seuil 0.3 / 0.5 / 0.7) :", nb_survivants_03, nb_survivants_05, nb_survivants_07)
print("Rappel (seuil 0.3 / 0.7) :", round(rappel_03, 2), round(rappel_07, 2))
# Seuil bas = on rate peu de survivants mais on en invente ; seuil haut = l'inverse. Le choix dépend du coût de chaque erreur.
```
</details>

## Exercice 9 ⭐⭐⭐ · Détective : trouve la fuite de données

`X_suspect` (chargé dans la Préparation) contient les 8 colonnes de `X` **plus** 3 colonnes mystère `Code_A`, `Code_B`, `Code_C`, qui seraient « issues d'un fichier retrouvé dans les archives ». L'une d'elles est une **fuite** : elle contient la réponse déguisée.
- `correlations` : corrélation de chaque colonne de `X_suspect` avec `y` (Series, triée)
- `colonne_fuite` : le nom de la colonne coupable
- `score_avec_fuite` : précision moyenne (cv=5) d'une `RandomForestClassifier(n_estimators=100, random_state=42)` sur `X_suspect`
- `score_sans_fuite` : la même chose après avoir retiré la colonne coupable

Résultat attendu : une corrélation > 0.8 pour la coupable (`Sex`, la vraie championne, plafonne à 0.54), un score qui passe de ≈ `0.91` à ≈ `0.82`.

<details><summary>Indice</summary>

`X_suspect.assign(Survived=y).corr()["Survived"].drop("Survived").sort_values()` — ou entraîne la forêt et regarde `feature_importances_`. Une variable que le modèle adore et qui n'existerait pas sur `test.csv`... c'est louche.
</details>

In [ ]:
# À toi
correlations = None
colonne_fuite = None
score_avec_fuite = None
score_sans_fuite = None

print(correlations)
print("Coupable :", colonne_fuite, "- score avec :", score_avec_fuite, "- sans :", score_sans_fuite)

In [ ]:
verifier("Exercice 9 · correlations pour les 11 colonnes", lambda: len(correlations) == 11 and "Code_B" in correlations.index)
verifier("Exercice 9 · colonne_fuite", colonne_fuite == "Code_B")
verifier("Exercice 9 · score_avec_fuite ≈ 0.91 : trop beau", proche(score_avec_fuite, 0.914, 0.03))
verifier("Exercice 9 · score_sans_fuite ≈ 0.82 : honnête", proche(score_sans_fuite, 0.82, 0.03))

<details><summary>Solution</summary>

```python
correlations = X_suspect.assign(Survived=y).corr()["Survived"].drop("Survived").sort_values()
colonne_fuite = correlations.abs().idxmax()          # Code_B : corrélation 0.85 avec la survie

foret_100 = RandomForestClassifier(n_estimators=100, random_state=42)
score_avec_fuite = cross_val_score(foret_100, X_suspect, y, cv=5).mean()
score_sans_fuite = cross_val_score(foret_100, X_suspect.drop(columns=[colonne_fuite]), y, cv=5).mean()

print(correlations.round(2))
print("Coupable :", colonne_fuite, "- score avec :", round(score_avec_fuite, 3), "- sans :", round(score_sans_fuite, 3))
# Code_B, c'est Survived avec 10 % de bruit. Sur test.csv cette colonne n'existerait pas : le 91 % est un mirage.
```
</details>

## Exercice 10 ⭐⭐⭐ · Un pipeline sklearn

Les k plus proches voisins mesurent des **distances** : `Fare` (jusqu'à 512) écrase `Sex` (0 ou 1). La solution propre est un **pipeline** qui met à l'échelle puis classe, en une seule étape :
- `score_brut` : précision moyenne (cv=5) de `KNeighborsClassifier(n_neighbors=5)` sur `X` tel quel
- `pipeline` : `make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5))`
- `score_pipeline` : sa précision moyenne (cv=5)

Résultat attendu : ≈ `0.72` brut contre ≈ `0.81` avec le pipeline. Bonus : le pipeline s'utilise comme n'importe quel modèle (`.fit`, `.predict`) et le scaler est refait proprement dans chaque paquet de la validation croisée.

<details><summary>Indice</summary>

`cross_val_score(pipeline, X, y, cv=5).mean()` : un pipeline se passe directement à `cross_val_score`.
</details>

In [ ]:
# À toi
score_brut = None
pipeline = None
score_pipeline = None

print("k-NN brut :", score_brut, "- avec pipeline :", score_pipeline)

In [ ]:
verifier("Exercice 10 · pipeline = scaler + k-NN", lambda: isinstance(pipeline, Pipeline) and isinstance(pipeline.steps[0][1], StandardScaler) and isinstance(pipeline.steps[-1][1], KNeighborsClassifier))
verifier("Exercice 10 · score_brut ≈ 0.72", proche(score_brut, 0.716, 0.03))
verifier("Exercice 10 · score_pipeline ≈ 0.81", proche(score_pipeline, 0.814, 0.03))
verifier("Exercice 10 · la mise à l'échelle gagne au moins 5 points", lambda: score_pipeline - score_brut > 0.05)

<details><summary>Solution</summary>

```python
score_brut = cross_val_score(KNeighborsClassifier(n_neighbors=5), X, y, cv=5).mean()
pipeline = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5))
score_pipeline = cross_val_score(pipeline, X, y, cv=5).mean()

print("k-NN brut :", round(score_brut, 3), "- avec pipeline :", round(score_pipeline, 3))
# StandardScaler ramène chaque colonne à moyenne 0 / écart-type 1 : une livre et un sexe pèsent enfin pareil.
```
</details>

## Exercice 11 ⭐⭐⭐ · Score trop beau : trouve pourquoi

Un binôme annonce fièrement **98 %** avec la cellule ci-dessous. C'est trop beau. Lis le code, trouve l'erreur, et réponds :
- `raison` : `"A"`, `"B"` ou `"C"`
  - A : une colonne de `X` contient la réponse (fuite de données)
  - B : le modèle est noté sur les passagers qu'il a vus pendant l'entraînement
  - C : la forêt sans limite de profondeur est simplement excellente
- `score_honnete` : le score que ce même modèle obtient **vraiment** sur des passagers jamais vus
- `ecart` : `score_annonce - score_honnete`

Résultat attendu : un `score_honnete` autour de `0.81` et un écart de plus de `0.10`.

<details><summary>Indice</summary>

Regarde sur quelles données `.score()` est appelé... et ce que contient `X_val`, qui n'est jamais utilisé.
</details>

In [ ]:
X_app, X_val, y_app, y_val = train_test_split(X, y, test_size=0.25, random_state=42)
modele_suspect = RandomForestClassifier(n_estimators=200, random_state=42).fit(X_app, y_app)
score_annonce = modele_suspect.score(X_app, y_app)
print(f"Score annoncé : {score_annonce*100:.1f} %")

# À toi
raison = None            # "A", "B" ou "C"
score_honnete = None
ecart = None

print("Raison :", raison, "- score honnête :", score_honnete, "- écart :", ecart)

In [ ]:
verifier("Exercice 11 · raison", raison == "B")
verifier("Exercice 11 · score_honnete ≈ 0.81 (mesuré sur X_val)", proche(score_honnete, 0.812, 0.03))
verifier("Exercice 11 · ecart > 0.10", lambda: proche(ecart, score_annonce - score_honnete, 1e-9) and ecart > 0.10)

<details><summary>Solution</summary>

```python
X_app, X_val, y_app, y_val = train_test_split(X, y, test_size=0.25, random_state=42)
modele_suspect = RandomForestClassifier(n_estimators=200, random_state=42).fit(X_app, y_app)
score_annonce = modele_suspect.score(X_app, y_app)
print(f"Score annoncé : {score_annonce*100:.1f} %")

raison = "B"             # .score(X_app, y_app) : on corrige le contrôle avec le corrigé sous les yeux
score_honnete = modele_suspect.score(X_val, y_val)
ecart = score_annonce - score_honnete

print("Raison :", raison, "- score honnête :", round(score_honnete, 3), "- écart :", round(ecart, 3))
# Une forêt sans max_depth apprend train par cœur (98 %) ; sur des passagers nouveaux elle retombe à 81 %.
```
</details>

## Exercice 12 ⭐⭐⭐ · Défi · la soumission finale, sans se tromper

Comme à la séance 7, on fabrique un faux `test.csv` : 100 passagers mis de côté (`faux_test`, sans `Survived`), leurs vraies réponses dans `reponses_cachees`, et `train_reduit` pour apprendre. À toi de produire une soumission **qui bat la règle bête** sur ce faux test :
1. `modele_final` : ton meilleur candidat (le gradient boosting de l'exercice 1 est un bon choix ; `recherche.best_estimator_` marche aussi), entraîné sur `preparer(train_reduit)` et `train_reduit["Survived"]`
2. `predictions` sur `preparer(faux_test)`, puis `submission_finale.csv` avec les colonnes `PassengerId` et `Survived`
3. `precision_finale` : proportion de bonnes prédictions sur `reponses_cachees`
4. `precision_bete_faux_test` : la précision de la règle « les femmes survivent » sur ces mêmes 100 passagers

Résultat attendu : un fichier de 100 lignes, `precision_bete_faux_test = 0.76` et `precision_finale` ≥ `0.77`.

<details><summary>Indice</summary>

C'est l'exercice 11 de la séance 7 avec un autre modèle. Pour la règle bête sur le faux test : `((faux_test["Sex"] == "female").astype(int).values == reponses_cachees.values).mean()`. Si ton modèle ne bat pas 0.76, essaie-en un autre : c'est exactement le travail d'une compétition.
</details>

In [ ]:
faux_test = train.sample(100, random_state=1)
reponses_cachees = faux_test["Survived"]
faux_test = faux_test.drop(columns=["Survived"])
train_reduit = train.drop(faux_test.index)

# À toi
modele_final = None
predictions = None
precision_finale = None
precision_bete_faux_test = None

print("Règle bête :", precision_bete_faux_test, "- ton modèle :", precision_finale)

In [ ]:
verifier("Exercice 12 · submission_finale.csv : 100 lignes, PassengerId et Survived",
         lambda: pd.read_csv("submission_finale.csv").shape == (100, 2) and list(pd.read_csv("submission_finale.csv").columns) == ["PassengerId", "Survived"])
verifier("Exercice 12 · les PassengerId sont ceux du faux test", lambda: list(pd.read_csv("submission_finale.csv")["PassengerId"]) == list(faux_test["PassengerId"]))
verifier("Exercice 12 · precision_bete_faux_test = 0.76", proche(precision_bete_faux_test, 0.76, 0.001))
verifier("Exercice 12 · ton modèle bat la règle bête (≥ 0.77)", lambda: proche(precision_finale, (pd.read_csv("submission_finale.csv")["Survived"].values == reponses_cachees.values).mean(), 1e-9) and precision_finale >= 0.77)

<details><summary>Solution</summary>

```python
faux_test = train.sample(100, random_state=1)
reponses_cachees = faux_test["Survived"]
faux_test = faux_test.drop(columns=["Survived"])
train_reduit = train.drop(faux_test.index)

modele_final = GradientBoostingClassifier(n_estimators=100, max_depth=3, random_state=42)
modele_final.fit(preparer(train_reduit), train_reduit["Survived"])
predictions = modele_final.predict(preparer(faux_test))

submission = pd.DataFrame({"PassengerId": faux_test["PassengerId"], "Survived": predictions})
submission.to_csv("submission_finale.csv", index=False)

precision_finale = (predictions == reponses_cachees.values).mean()
precision_bete_faux_test = ((faux_test["Sex"] == "female").astype(int).values == reponses_cachees.values).mean()

print("Règle bête :", precision_bete_faux_test, "- ton modèle :", precision_finale)
# 0.80 contre 0.76 : 4 passagers de mieux sur 100. Sur Kaggle, c'est l'écart entre 0.766 et 0.80 au classement.
```
</details>

## Bravo !

Tu sais maintenant comparer des modèles proprement, expliquer *pourquoi* l'un décide mieux, repérer le par cœur sur une courbe, régler des boutons avec `GridSearchCV`, choisir un seuil, et surtout **te méfier d'un score trop beau** : fuite de données ou note sur ses propres données.

**Pour aller plus loin** : refais l'exercice 12 avec le vrai `test.csv` de Kaggle et soumets `submission_finale.csv`. Puis remplace `preparer` par une version avec `Prix_par_personne` (séance 7, exercice 8) et regarde si le score bouge au classement : https://www.kaggle.com/competitions/titanic/leaderboard